# Task 2: Subindustry Classification
### FLANG-BERT | v9c+ Sibling Context | Aux Heads | 4CE+4FL+4CE
**Fixes applied**: SubIndustry column name, FL_GAMMA=1.0, GroupShuffleSplit data loading,
sibling descriptions, hierarchical prefix tokens `[sector] [industry]`

## 0. Install

In [ ]:
!pip install -q transformers==4.40.0 accelerate scikit-learn


## 1. Imports

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report
from torch.cuda.amp import autocast, GradScaler

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Config

In [ ]:
class Config:
    MODEL_NAME    = 'SALT-NLP/FLANG-BERT'
    MAX_LEN       = 512
    # Filled automatically after LabelEncoder.fit()
    NUM_SUBIND    = None
    NUM_INDUSTRY  = None
    NUM_SECTOR    = None
    # Auxiliary loss weights (proven in Task 1)
    W_SUBIND      = 1.00
    W_INDUSTRY    = 0.15
    W_SECTOR      = 0.20
    # Schedule: 4CE -> 4FL -> 4CE
    CE_EPOCHS_1   = 4
    FL_EPOCHS     = 4
    FL_GAMMA      = 1.0   # Fix: 2.0 always hurt in Task 1 experiments
    CE_EPOCHS_2   = 4
    # Optimiser
    BATCH_SIZE    = 16
    GRAD_ACCUM    = 2       # effective batch = 32
    LR            = 2e-5
    WEIGHT_DECAY  = 0.01
    WARMUP_RATIO  = 0.06
    MAX_GRAD_NORM = 1.0
    # Paths
    BASE_DIR      = Path('/content/drive/MyDrive/CAPSTONE')
    RAW_DIR       = BASE_DIR / 'raw'
    OUTPUT_DIR    = Path('/content/task2_flangbert')
    RAW_CSV       = RAW_DIR / 'task2_subindustry_classification_final.csv'
    SEED          = 42

cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print('Config OK — FL_GAMMA:', cfg.FL_GAMMA)


## 3. Mount Drive & Load Raw Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Fix: column is 'SubIndustry' (capital I) — not 'Subindustry'
t2 = pd.read_csv(cfg.RAW_CSV,
                 dtype={'SubIndustry': str, 'CompanyId': str})
t2['AsOfDate'] = pd.to_datetime(t2['AsOfDate'], errors='coerce')

print(f'Raw rows: {len(t2):,}')
print(f'Columns : {list(t2.columns)}')
print(f'Unique SubIndustry: {t2["SubIndustry"].nunique()}')
print(f'Unique CompanyId  : {t2["CompanyId"].nunique()}')
t2.head(3)


## 4. Text Normalisation + v9c+ Builder

In [ ]:
def norm(text):
    """Clean text: unicode artifacts, extra whitespace, lowercase."""
    if pd.isna(text): return ''
    text = str(text)
    text = text.replace('\u201c', ' ').replace('\u201d', ' ')
    text = text.replace('&', ' and ')
    text = ''.join(c if ord(c) < 128 else ' ' for c in text)
    return ' '.join(text.split()).lower().strip()

t2['SegmentName']        = t2['SegmentName'].apply(norm)
t2['SegmentDescription'] = t2['SegmentDescription'].apply(norm)
# Fallback: use SegmentName when description is empty
t2['SegmentDescription'] = t2.apply(
    lambda r: r['SegmentName'] if not r['SegmentDescription'].strip()
    else r['SegmentDescription'], axis=1)

print('Normalisation done.')
print(t2[['SegmentName', 'SegmentDescription']].head(3).to_string())


In [ ]:
def build_v9c_text(row, company_df):
    """
    v9c+ format with two fixes applied:
      1. Hierarchical prefix tokens [sector_3] [industry_8] — same trick as
         Task 1 champion that gave 0.7380 at epoch 1.
      2. Siblings use SegmentDescription (first 25 words) not just name —
         richer context for distinguishing subindustries.

    Format:
      '[311] [31130010] [PRIMARY] segment name: description
       [SIBLINGS] sib1 first 25 words | sib2 first 25 words'
    """
    sub_code   = str(row['SubIndustry']).strip()   # Fix: capital I
    sector_3   = sub_code[:3]
    industry_8 = sub_code[:8]
    seg_name   = str(row['SegmentName']).strip()
    seg_desc   = str(row['SegmentDescription']).strip()

    # Hierarchical prefix (proven gain in Task 1)
    text = f'[{sector_3}] [{industry_8}] [PRIMARY] {seg_name}: {seg_desc}'

    # Siblings: use SegmentDescription (25 words) for richer sibling signal
    sibs = company_df[
        (company_df['CompanyId'] == row['CompanyId']) &
        (company_df.index != row.name)
    ]
    sib_parts = []
    for _, sib in sibs.iterrows():
        sib_desc  = str(sib['SegmentDescription']).strip()
        sib_short = ' '.join(sib_desc.split()[:25])
        sib_parts.append(sib_short)
    if sib_parts:
        text += ' [SIBLINGS] ' + ' | '.join(sib_parts[:5])

    return text


print('Building v9c+ texts (may take ~1-2 min for large datasets)...')
t2['text']         = t2.apply(lambda r: build_v9c_text(r, t2), axis=1)
t2['IndustryCode'] = t2['SubIndustry'].str[:8]   # Fix: SubIndustry capital I
t2['SectorCode']   = t2['SubIndustry'].str[:3]

print(f'Texts built. Sample:')
print(t2['text'].iloc[0][:400])


## 5. GroupShuffleSplit (company-level, no leakage)

In [ ]:
# Fix: no separate train/val/test CSVs — split raw file by CompanyId group
# GroupShuffleSplit ensures all segments of a company stay in same split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=cfg.SEED)
train_idx, val_idx = next(splitter.split(
    t2['text'], t2['SubIndustry'], groups=t2['CompanyId']
))

train_df = t2.iloc[train_idx].copy().reset_index(drop=False)
val_df   = t2.iloc[val_idx].copy().reset_index(drop=False)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
print(f'Train companies: {train_df["CompanyId"].nunique():,}')
print(f'Val companies  : {val_df["CompanyId"].nunique():,}')
# Verify no company appears in both splits
overlap = set(train_df['CompanyId']) & set(val_df['CompanyId'])
print(f'Company leakage: {len(overlap)} (must be 0)')
print(f'Unique SubIndustries in train: {train_df["SubIndustry"].nunique()}')


## 6. Label Encoders + Class Weights

In [ ]:
# Fix: SubIndustry (capital I) everywhere
le_sub = LabelEncoder().fit(train_df['SubIndustry'])
le_ind = LabelEncoder().fit(train_df['IndustryCode'])
le_sec = LabelEncoder().fit(train_df['SectorCode'])

cfg.NUM_SUBIND   = len(le_sub.classes_)
cfg.NUM_INDUSTRY = len(le_ind.classes_)
cfg.NUM_SECTOR   = len(le_sec.classes_)
print(f'SubInd: {cfg.NUM_SUBIND} | Ind: {cfg.NUM_INDUSTRY} | Sec: {cfg.NUM_SECTOR}')

for df in [train_df, val_df]:
    df['label_sub'] = le_sub.transform(df['SubIndustry'])    # Fix: capital I
    df['label_ind'] = le_ind.transform(df['IndustryCode'])
    df['label_sec'] = le_sec.transform(df['SectorCode'])

# Handle val labels unseen in train (map to nearest or ignore)
val_unseen = set(val_df['SubIndustry'].unique()) - set(le_sub.classes_)
if val_unseen:
    print(f'WARNING: {len(val_unseen)} SubIndustry classes in val not seen in train.')
    print('These will be filtered from val evaluation.')
    val_df_eval = val_df[val_df['SubIndustry'].isin(le_sub.classes_)].copy()
else:
    val_df_eval = val_df.copy()
    print('All val SubIndustry labels seen in train.')

# Inverse-frequency class weights
counts = np.bincount(train_df['label_sub'], minlength=cfg.NUM_SUBIND).astype(float)
counts = np.where(counts == 0, 1, counts)
w = 1.0 / counts
w = w / w.sum() * cfg.NUM_SUBIND
class_weights_t = torch.tensor(w, dtype=torch.float32).to(device)

vc = train_df['SubIndustry'].value_counts()
print(f'Imbalance ratio: {vc.max()/max(vc.min(),1):.0f}x | classes < 10 samples: {(vc<10).sum()}')


## 7. Dataset & Balanced Sampler

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)

# Add special tokens used in v9c+ format
special_tokens = ['[PRIMARY]', '[SIBLINGS]']
tokenizer.add_tokens(special_tokens, special_tokens=True)
print(f'Vocab size after adding tokens: {len(tokenizer)}')


class SegDataset(Dataset):
    def __init__(self, df, tok, max_len, is_test=False):
        self.texts   = df['text'].tolist()
        self.tok     = tok
        self.max_len = max_len
        self.is_test = is_test
        if not is_test:
            self.ls = df['label_sub'].tolist()
            self.li = df['label_ind'].tolist()
            self.lc = df['label_sec'].tolist()

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}
        if not self.is_test:
            item['label_sub'] = torch.tensor(self.ls[i], dtype=torch.long)
            item['label_ind'] = torch.tensor(self.li[i], dtype=torch.long)
            item['label_sec'] = torch.tensor(self.lc[i], dtype=torch.long)
        return item


train_ds   = SegDataset(train_df,      tokenizer, cfg.MAX_LEN)
val_ds     = SegDataset(val_df_eval,   tokenizer, cfg.MAX_LEN)

# WeightedRandomSampler: oversample rare subindustries per batch
samp_w = w[train_df['label_sub'].values]
sampler = WeightedRandomSampler(
    weights=torch.tensor(samp_w, dtype=torch.double),
    num_samples=len(train_ds), replacement=True
)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,   sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE*2, shuffle=False,   num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## 8. Model — FLANG-BERT + 3 Heads

In [ ]:
class FlangBERT(nn.Module):
    """
    FLANG-BERT backbone + 3 multi-class heads.
    Task 2 = multi-CLASS (one label per segment), CrossEntropyLoss.
      head_sub : cfg.NUM_SUBIND   classes  (primary, weighted CE)
      head_ind : cfg.NUM_INDUSTRY classes  (aux, w=0.15)
      head_sec : cfg.NUM_SECTOR   classes  (aux, w=0.20)
    """
    def __init__(self, model_name, n_sub, n_ind, n_sec, vocab_size, dropout=0.1):
        super().__init__()
        self.enc  = AutoModel.from_pretrained(model_name)
        # Resize for added special tokens ([PRIMARY], [SIBLINGS])
        self.enc.resize_token_embeddings(vocab_size)
        h = self.enc.config.hidden_size   # 768
        self.norm  = nn.LayerNorm(h)
        self.drop  = nn.Dropout(dropout)
        self.h_sub = nn.Linear(h, n_sub)
        self.h_ind = nn.Linear(h, n_ind)
        self.h_sec = nn.Linear(h, n_sec)

    def forward(self, input_ids, attention_mask):
        cls = self.enc(input_ids=input_ids,
                       attention_mask=attention_mask).last_hidden_state[:, 0]
        cls = self.drop(self.norm(cls))
        return self.h_sub(cls), self.h_ind(cls), self.h_sec(cls)


model = FlangBERT(
    cfg.MODEL_NAME,
    cfg.NUM_SUBIND, cfg.NUM_INDUSTRY, cfg.NUM_SECTOR,
    vocab_size=len(tokenizer)
).to(device)

print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'Vocab size (with special tokens): {len(tokenizer)}')


## 9. Loss Functions (multi-class CE, NOT BCE)

In [ ]:
class FocalLoss(nn.Module):
    """Multi-class Focal Loss. gamma=1.0 (proven better than 2.0 in Task 1)."""
    def __init__(self, gamma=1.0, weight=None):   # Fix: gamma=1.0
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        loss = (1 - torch.exp(-ce)) ** self.gamma * ce
        return loss.mean()


ce_fn    = nn.CrossEntropyLoss(weight=class_weights_t)
focal_fn = FocalLoss(gamma=cfg.FL_GAMMA, weight=class_weights_t)  # gamma=1.0
aux_fn   = nn.CrossEntropyLoss()


def total_loss(ls, li, lc, y_s, y_i, y_c, primary):
    """primary(subind) + 0.15*CE(industry) + 0.20*CE(sector)"""
    return (cfg.W_SUBIND   * primary(ls, y_s) +
            cfg.W_INDUSTRY * aux_fn(li, y_i) +
            cfg.W_SECTOR   * aux_fn(lc, y_c))

print(f'Focal gamma: {cfg.FL_GAMMA}  (1.0, not 2.0)')


## 10. Training Utilities

In [ ]:
def make_opt_sched(model, n_steps, lr):
    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n,p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)], 'weight_decay': cfg.WEIGHT_DECAY},
        {'params': [p for n,p in model.named_parameters()
                    if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    opt   = torch.optim.AdamW(params, lr=lr)
    sched = get_cosine_schedule_with_warmup(
        opt, int(cfg.WARMUP_RATIO * n_steps), n_steps)
    return opt, sched


def train_epoch(model, loader, opt, sched, scaler, loss_fn):
    model.train()
    opt.zero_grad()
    total = 0.0
    for step, b in enumerate(loader):
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        ys  = b['label_sub'].to(device)
        yi  = b['label_ind'].to(device)
        yc  = b['label_sec'].to(device)
        with autocast():
            ls, li, lc = model(ids, msk)
            loss = total_loss(ls, li, lc, ys, yi, yc, loss_fn) / cfg.GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        total += loss.item() * cfg.GRAD_ACCUM
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for b in loader:
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        with autocast():
            ls, _, _ = model(ids, msk)
        preds.extend(ls.argmax(-1).cpu().numpy())
        labels.extend(b['label_sub'].numpy())
    return f1_score(labels, preds, average='macro', zero_division=0), preds, labels

print('Utilities ready.')


## 11. Phase 1 — CrossEntropy Warm-Up (4 epochs)

In [ ]:
steps1 = (len(train_loader) // cfg.GRAD_ACCUM) * cfg.CE_EPOCHS_1
opt, sched = make_opt_sched(model, steps1, cfg.LR)
scaler = GradScaler()
best_f1, log = 0.0, []

print('=== Phase 1: CE warm-up ===')
for ep in range(cfg.CE_EPOCHS_1):
    loss = train_epoch(model, train_loader, opt, sched, scaler, ce_fn)
    vf1, _, _ = evaluate(model, val_loader)
    log.append({'phase': 'CE1', 'epoch': ep+1, 'loss': loss, 'val_f1': vf1})
    print(f'  [{ep+1}/{cfg.CE_EPOCHS_1}] loss={loss:.4f}  val_macro_F1={vf1:.4f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), cfg.OUTPUT_DIR / 'best.pt')
        print(f'    ✓ New best {best_f1:.4f}')


## 12. Phase 2 — Focal Loss Hard-Mining (4 epochs, gamma=1.0)

In [ ]:
steps2 = (len(train_loader) // cfg.GRAD_ACCUM) * cfg.FL_EPOCHS
opt, sched = make_opt_sched(model, steps2, cfg.LR * 0.5)
scaler = GradScaler()

print(f'=== Phase 2: Focal Loss (gamma={cfg.FL_GAMMA}) hard-mining ===')
for ep in range(cfg.FL_EPOCHS):
    loss = train_epoch(model, train_loader, opt, sched, scaler, focal_fn)
    vf1, _, _ = evaluate(model, val_loader)
    log.append({'phase': 'FL', 'epoch': ep+1, 'loss': loss, 'val_f1': vf1})
    print(f'  [{ep+1}/{cfg.FL_EPOCHS}] loss={loss:.4f}  val_macro_F1={vf1:.4f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), cfg.OUTPUT_DIR / 'best.pt')
        print(f'    ✓ New best {best_f1:.4f}')


## 13. Phase 3 — CrossEntropy Fine-Tune (4 epochs)

In [ ]:
steps3 = (len(train_loader) // cfg.GRAD_ACCUM) * cfg.CE_EPOCHS_2
opt, sched = make_opt_sched(model, steps3, cfg.LR * 0.3)
scaler = GradScaler()

print('=== Phase 3: CE fine-tune ===')
for ep in range(cfg.CE_EPOCHS_2):
    loss = train_epoch(model, train_loader, opt, sched, scaler, ce_fn)
    vf1, _, _ = evaluate(model, val_loader)
    log.append({'phase': 'CE2', 'epoch': ep+1, 'loss': loss, 'val_f1': vf1})
    print(f'  [{ep+1}/{cfg.CE_EPOCHS_2}] loss={loss:.4f}  val_macro_F1={vf1:.4f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), cfg.OUTPUT_DIR / 'best.pt')
        print(f'    ✓ New best {best_f1:.4f}')


## 14. Final Evaluation + Per-Class Report

In [ ]:
model.load_state_dict(torch.load(cfg.OUTPUT_DIR / 'best.pt'))
final_f1, preds, labels = evaluate(model, val_loader)
print(f'Best Val Macro-F1: {final_f1:.4f}')

report = classification_report(labels, preds,
    target_names=le_sub.classes_, zero_division=0, output_dict=True)
rdf = pd.DataFrame(report).T.iloc[:-3]
rdf.to_csv(cfg.OUTPUT_DIR / 'per_class_f1.csv')

zero_f1 = (rdf['f1-score'] == 0).sum()
print(f'Zero-F1 classes : {zero_f1} / {cfg.NUM_SUBIND}')
print(f'Theoretical max : {(cfg.NUM_SUBIND - zero_f1)/cfg.NUM_SUBIND:.4f}  macro-F1')

pd.DataFrame(log).to_csv(cfg.OUTPUT_DIR / 'training_log.csv', index=False)
print()
print(pd.DataFrame(log).to_string())


## 15. Test Inference → submission.csv

In [ ]:
# If you have a held-out test CSV without labels:
# test_raw = pd.read_csv(cfg.RAW_DIR / 'task2_test.csv', dtype={'SubIndustry': str, 'CompanyId': str})
# test_raw['SegmentName']        = test_raw['SegmentName'].apply(norm)
# test_raw['SegmentDescription'] = test_raw['SegmentDescription'].apply(norm)
# test_raw['text'] = test_raw.apply(lambda r: build_v9c_text(r, test_raw), axis=1)
# test_ds = SegDataset(test_raw, tokenizer, cfg.MAX_LEN, is_test=True)

# For now: run inference on val set as sanity check
@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for b in loader:
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        with autocast():
            ls, _, _ = model(ids, msk)
        out.extend(ls.argmax(-1).cpu().numpy())
    return out

val_df_eval['PredictedSubIndustry'] = le_sub.inverse_transform(predict(model, val_loader))
val_df_eval[['CompanyId', 'AsOfDate', 'SegmentName',
             'SubIndustry', 'PredictedSubIndustry']].to_csv(
    cfg.OUTPUT_DIR / 'val_predictions.csv', index=False)
print(f'Saved val_predictions.csv — {len(val_df_eval):,} rows')
val_df_eval[['SegmentName', 'SubIndustry', 'PredictedSubIndustry']].head(10)


## 16. Diagnostics — What Is Dragging F1 Down

In [ ]:
print('=== Bottom 20 classes by F1 (biggest drag on macro-F1) ===')
print(rdf.sort_values('f1-score').head(20)[['f1-score', 'support']].to_string())

corr = rdf[['f1-score', 'support']].corr().loc['f1-score', 'support']
print(f'\nCorr(support, F1) = {corr:.3f}')
if corr > 0.5:
    print('-> Imbalance is primary bottleneck: augment rare classes')
elif corr < 0.3:
    print('-> Label confusion is primary bottleneck: review ambiguous subindustry pairs')
else:
    print('-> Both imbalance and label confusion present')


## 17. Optional: Temperature Scaling (+0.01–0.02 F1, no retraining)

In [ ]:
@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    lg_list, lb_list = [], []
    for b in loader:
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        with autocast():
            ls, _, _ = model(ids, msk)
        lg_list.append(ls.cpu())
        lb_list.append(b['label_sub'])
    return torch.cat(lg_list), torch.cat(lb_list)


vl, vlab = get_logits(model, val_loader)
best_T, best_Tf1 = 1.0, 0.0
for T in np.arange(0.5, 3.1, 0.1):
    p = (vl / T).argmax(-1).numpy()
    f = f1_score(vlab.numpy(), p, average='macro', zero_division=0)
    if f > best_Tf1:
        best_T, best_Tf1 = T, f

print(f'Best T = {best_T:.2f}')
print(f'Val macro-F1 after calibration : {best_Tf1:.4f}  (was {final_f1:.4f})')
print(f'Gain : {best_Tf1 - final_f1:+.4f}')
